In [0]:
df = spark.read.format("delta").load("/Volumes/workspace/default/adsb_data/bronze/live_states")

In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, DoubleType, BooleanType,
    IntegerType, ArrayType, DateType, TimestampType
)

expected_schema = StructType([
    StructField("icao24", StringType(), True),
    StructField("callsign", StringType(), True),
    StructField("origin_country", StringType(), True),
    StructField("time_position", LongType(), True),
    StructField("last_contact", LongType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("baro_altitude", DoubleType(), True),
    StructField("on_ground", BooleanType(), True),
    StructField("velocity", DoubleType(), True),
    StructField("true_track", DoubleType(), True),
    StructField("vertical_rate", DoubleType(), True),
    StructField("sensors", ArrayType(IntegerType()), True),
    StructField("geo_altitude", DoubleType(), True),
    StructField("squawk", StringType(), True),
    StructField("spi", BooleanType(), True),
    StructField("position_source", IntegerType(), True),
    StructField("category", IntegerType(), True),
    StructField("api_timestamp", LongType(), True),
    StructField("ingested_at", StringType(), True),
    StructField("ingest_date", DateType(), True),
    StructField("ingest_hour", IntegerType(), True),
])

assert df.schema == expected_schema, f"Schema mismatch: {df.schema}"

In [0]:
#this code is mainly for testing (the data colleected when dev didn't contain all the conditions we'd hope to see so we create mock data for these events for donwstream)
from datetime import date

mock_rows = [
    # Emergency squawk mock data
    ("4ca2b1", "BAW291", "United Kingdom", 1775058950, 1775058950, -0.4614, 51.4775, 8000.0, False, 250.0, 90.0, 0.0, None, 7900.0, "7700", False,  0, 0, 1775058950, "2025-01-01T00:00:00Z", date(2025, 1, 1), 0),
    ("3f8a12", "AFR772", "France",         1775058950, 1775058950, 2.5478,  49.0097, 9000.0, False, 280.0, 180.0, 0.0, None, 8900.0, "7600",    False, 0, 0, 1775058950, "2025-01-01T00:00:00Z", date(2025, 1, 1), 0),
    ("a1b2c3", "DAL451", "United States",  1775058950, 1775058950, -73.778, 40.641,  7000.0, False, 260.0, 270.0, 0.0, None, 6900.0, "7500",    False, 0, 0, 1775058950, "2025-01-01T00:00:00Z", date(2025, 1, 1), 0),

    # Rapid altitude drop mock data
    ("4d3a1b", "DLH441", "Germany", 1775058950, 1775058950, 8.6821, 50.0379, 8000.0, False, 250.0, 90.0, 0.0, None, 7900.0, "1234", False, 0, 0,    1775058950, "2025-01-01T10:00:00Z", date(2025, 1, 1), 10),
    ("4d3a1b", "DLH441", "Germany", 1775058980, 1775058980, 8.6821, 50.0379, 1200.0, False, 250.0, 90.0, 0.0, None, 1100.0, "1234", False, 0, 0,    1775058980, "2025-01-01T10:00:00Z", date(2025, 1, 1), 10),

    # Extreme vertical rate mock data
    ("4b7f2a", "RYR892", "Ireland", 1775058950, 1775058950, -6.2603, 53.3498, 7000.0, False, 230.0, 90.0, 45.0, None, 6900.0, "1234", False, 0, 0,  1775058950, "2025-01-01T14:00:00Z", date(2025, 1, 1), 14),
    ("3a9c4e", "EZY331", "France",   1775058950, 1775058950, 2.5478,  49.0097, 5000.0, False, 210.0, 180.0, -30.0, None, 4900.0, "1234", False, 0, 0, 1775058950, "2025-01-01T08:00:00Z", date(2025, 1, 1), 8),

    # Unusual speed mock data
    ("a3f1c7", "UAL789", "United States", 1775058950, 1775058950, -87.9048, 41.9742, 10000.0, False, 400.0, 90.0,  0.0, None, 9900.0, "1234",   False, 0, 0, 1775058950, "2025-01-01T12:00:00Z", date(2025, 1, 1), 12),
    ("3c6b2e", "AFR991", "France",        1775058950, 1775058950, 2.5478,  49.0097,  7000.0, False,  30.0, 180.0, 0.0, None, 6900.0, "1234",    False, 0, 0, 1775058950, "2025-01-01T16:00:00Z", date(2025, 1, 1), 16),

    # Signal gap mock data
    ("4e2b1a", "BAW442", "United Kingdom", 1775058950, 1775058950, -0.4614, 51.4775, 5000.0, False, 250.0, 90.0, 0.0, None, 4900.0, "1234", False,  0, 0, 1775058950, "2025-01-01T09:00:00Z", date(2025, 1, 1), 9),
    ("4e2b1a", "BAW442", "United Kingdom", 1775059550, 1775059550, -0.4614, 51.4775, 5000.0, False, 250.0, 90.0, 0.0, None, 4900.0, "1234", False,  0, 0, 1775059550, "2025-01-01T09:10:00Z", date(2025, 1, 1), 9),
]

mock_df = spark.createDataFrame(mock_rows, schema=expected_schema)
df = df.union(mock_df)

In [0]:
from pyspark.sql.functions import current_timestamp, col, lit, when, concat
df = df.withColumn("detected_at", current_timestamp())

In [0]:
emergency_squawk_df = df.dropna(subset=["squawk"]).filter(col("squawk").isin(["7500", "7600", "7700"]))\
        .withColumn("anomaly_type", lit("EMERGENCY_SQUAWK")) \
        .withColumn("anomaly_severity", lit("CRITICAL")) \
        .withColumn("detail", when(col("squawk") == "7500", lit("Squawk 7500 detected — Hijack in progress"))
                              .when(col("squawk") == "7600", lit("Squawk 7600 detected — Radio communication failure"))
                                .otherwise(lit("Squawk 7700 detected — General emergency declared") )
        )

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag

window_spec = Window.partitionBy("icao24").orderBy("api_timestamp")
rapid_altitude_drop_df = df.dropna(subset=["baro_altitude", "api_timestamp"]).withColumn("prev_baro_altitude",lag("baro_altitude",1).over(window_spec)).\
        withColumn("prev_api_timestamp", lag("api_timestamp",1).over(window_spec))

In [0]:

BARO_ALTITUDE_DROP_THRESHOLD = -500
MIN_ARIBORN_BARO_ALTITUDE = 1000
rapid_altitude_drop_df = rapid_altitude_drop_df.filter( (col("baro_altitude") - col("prev_baro_altitude") < BARO_ALTITUDE_DROP_THRESHOLD) &
                                                       ( col("on_ground") == False) &
                                                       ( col("baro_altitude") > MIN_ARIBORN_BARO_ALTITUDE) )\
        .withColumn("anomaly_type", lit("RAPID_ALTITUDE_DROP")) \
        .withColumn("anomaly_severity", lit("HIGH")) \
        .withColumn("detail", concat(
                                    lit("Rapid altitude drop detected — from "),
                                    col("prev_baro_altitude").cast("string"),
                                    lit("m to "),
                                    col("baro_altitude").cast("string"),
                                    lit("m")
                            )
                    )

In [0]:
from pyspark.sql.functions import abs as abs_spark

EXTREME_VERTICAL_RATE_CRITICAL_THRESHOLD = 40
EXTREME_VERTICAL_RATE_HIGH_THRESHOLD = 25

extreme_vertical_rate_df = df.dropna(subset=["vertical_rate"]).withColumn("abs_vertical_rate", abs_spark(col("vertical_rate")))\
                            .filter( (col("abs_vertical_rate") > EXTREME_VERTICAL_RATE_HIGH_THRESHOLD) & (col("on_ground") ==False) )\
                            .withColumn("anomaly_type", lit("EXTREME_VERTICAL_RATE")) \
                            .withColumn("anomaly_severity", when(
                                                                col("abs_vertical_rate") > EXTREME_VERTICAL_RATE_CRITICAL_THRESHOLD,
                                                                    lit("CRITICAL")
                                                                )
                                                            .otherwise(lit("HIGH"))
                            )\
                            .withColumn("detail", concat(
                                                    lit("Extreme vertical rate detected — "),
                                                    col("abs_vertical_rate").cast("string"),
                                                    lit("m/s")
                                                    )
                                )\
                            .drop("abs_vertical_rate")

In [0]:
MAX_SPEED_THRESHOLD = 350
MIN_SPEED_HIGH_ALTITUDE_THRESHOLD = 50
MIN_HIGH_ALTITUDE = 6000

unusual_speed_df = df.dropna(subset=["velocity", "baro_altitude"]).filter(
                                                                ( col("on_ground") == False) & ( ( col("velocity") > MAX_SPEED_THRESHOLD) 
                                                                                        | ( 
                                                                                        (col("baro_altitude") > MIN_HIGH_ALTITUDE) & (col("velocity") < MIN_SPEED_HIGH_ALTITUDE_THRESHOLD))
                                                                                    ) 
                    ).withColumn("anomaly_type", lit("UNUSUAL_SPEED")) \
                    .withColumn("anomaly_severity", lit("MEDIUM"))\
                    .withColumn("detail", 
                                        when(col("velocity") > MAX_SPEED_THRESHOLD, concat( 
                                                                                        lit("Unusual speed detected — velocity "),
                                                                                        col("velocity").cast("string"),
                                                                                        lit(" m/s exceeds maximum threshold")
                                                                                    )
                                        ).otherwise(concat(
                                                    lit("Unusual speed detected — velocity "),
                                                    col("velocity").cast("string"),
                                                    lit(" m/s too low at "),
                                                    col("baro_altitude").cast("string"),
                                                    lit("m altitude")
                                                )
                                        )
                    )

In [0]:
MIN_SIGNAL_GAP_SECONDS = 300 #5min
MAX_SIGNAL_GAP_SECONDS = 3600 #60min

signal_gap_df = df.withColumn("prev_api_timestamp",lag("api_timestamp",1).over(window_spec))

signal_gap_df = signal_gap_df.dropna(subset=["api_timestamp"]).filter(
                                    ( col("api_timestamp") - col("prev_api_timestamp") > MIN_SIGNAL_GAP_SECONDS ) & ( col("api_timestamp") - col("prev_api_timestamp") < MAX_SIGNAL_GAP_SECONDS) & (col("on_ground") == False))\
                            .withColumn("anomaly_type", lit("SIGNAL_GAP")) \
                            .withColumn("anomaly_severity", lit("MEDIUM"))\
                            .withColumn("detail",concat(lit("Signal gap detected — "),
                                            (col("api_timestamp") - col("prev_api_timestamp")).cast("string"),
                                            lit(" since last contact")
                                            )
                            )



In [0]:
COLUMNS_TO_SELECT = ["icao24", "callsign", "latitude", "longitude", "baro_altitude", "api_timestamp", "anomaly_type", "anomaly_severity", "detail", "detected_at"]

anomaly_schema = StructType([
    StructField("icao24", StringType(), True),
    StructField("callsign", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("baro_altitude", DoubleType(), True),
    StructField("api_timestamp", LongType(), True),
    StructField("anomaly_type", StringType(), True),
    StructField("anomaly_severity", StringType(), True),
    StructField("detail", StringType(), True),
    StructField("detected_at", TimestampType(), True),
])

anomaly_df = spark.createDataFrame([],schema=anomaly_schema)

emergency_squawk_df = emergency_squawk_df.select(COLUMNS_TO_SELECT)
rapid_altitude_drop_df = rapid_altitude_drop_df.select(COLUMNS_TO_SELECT)
extreme_vertical_rate_df = extreme_vertical_rate_df.select(COLUMNS_TO_SELECT)
unusual_speed_df = unusual_speed_df.select(COLUMNS_TO_SELECT)
signal_gap_df = signal_gap_df.select(COLUMNS_TO_SELECT)

anomaly_df = anomaly_df.union(emergency_squawk_df).union(rapid_altitude_drop_df).union(extreme_vertical_rate_df).union(unusual_speed_df).union(signal_gap_df)



In [0]:

silver_path = "/Volumes/workspace/default/adsb_data/silver/anomalies"

anomaly_df.write.format("delta").mode("overwrite").save(silver_path)